# HelixIR — GPU Benchmark (Colab), fully instrumented

Every cell prints a diagnostic and swallows nothing — if something can't run on
this GPU it says **why** instead of silently falling back. Run **Runtime → Run all**
on a GPU runtime, then paste the whole output back.

What you get:
1. Environment + GPU capability report
2. Analytic baseline (always runs)
3. **vLLM**: import diagnostic → smoke test → real serving benchmark vs. analytic
4. **Triton** flash-attention: correctness (PASS/FAIL) + fair fp16-vs-fp16 speed
5. **Sweep**: fp16 (does an 8B fit?) and fp8 (find the fits→spills transition)

> Needs the pushed `helix/inference` + `attention_triton` code on `BRANCH` below.

## 0 · GPU + clone + install

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,compute_cap --format=csv || echo 'NO GPU — Runtime → Change runtime type → GPU'

In [ ]:
REPO   = 'https://github.com/HackathonGroupMulti/HelixIR.git'
BRANCH = 'main'

import os
if not os.path.isdir('HelixIR'):
    !git clone --branch {BRANCH} --depth 1 {REPO}
%cd HelixIR
assert os.path.isdir('helix/inference'), 'helix/inference missing — push it or fix BRANCH'
assert os.path.exists('helix/kernels/attention_triton.py'), 'attention_triton.py missing'
print('HelixIR checkout OK on', BRANCH)

In [ ]:
!pip -q install -e .
!pip -q install --upgrade 'jax[cuda12]'
# vLLM is a large install (several minutes). If it fails, section 3 will report why.
!pip -q install vllm 2>&1 | tail -5
print('installs attempted')

In [ ]:
# Environment report
import torch
print('torch      :', torch.__version__)
print('cuda avail :', torch.cuda.is_available())
if torch.cuda.is_available():
    print('gpu        :', torch.cuda.get_device_name(0))
    print('capability :', torch.cuda.get_device_capability(0))
try:
    import jax; print('jax        :', jax.__version__, '| devices:', [d.platform for d in jax.devices()])
except Exception as e:
    print('jax        : ERROR', e)

def helix_device():
    if not torch.cuda.is_available():
        return 'CPU'
    name = torch.cuda.get_device_name(0).upper().replace(' ', '')
    for key in ['H100','A100','L4','T4','V100','RTX4090']:
        if key.replace(' ','') in name:
            return key.replace('RTX4090','RTX 4090')
    return 'A100'

DEVICE   = helix_device()
CC       = torch.cuda.get_device_capability(0) if torch.cuda.is_available() else (0,0)
AMPERE   = CC >= (8,0)   # Pallas flash-attn needs Ampere+; Triton also runs on Turing (T4)
print('\nHelixIR device =', DEVICE, '| cc =', CC, '| ampere+ =', AMPERE)

## 1 · Analytic baseline (prediction — always runs)

In [ ]:
from helix.inference import ModelConfig, analyze_inference, print_inference_report
print_inference_report(analyze_inference(ModelConfig.from_preset('llama3-8b'),
                       batch=16, prompt_len=2048, gen_len=128, device=DEVICE))

## 2 · vLLM — diagnostic, smoke test, then real serving

Three steps so we can see exactly where (if anywhere) vLLM fails on this GPU.

In [ ]:
# 2a · Can we even import vLLM?
VLLM_OK = False
try:
    import vllm
    VLLM_OK = True
    print('vLLM import OK, version', vllm.__version__)
except Exception:
    import traceback; traceback.print_exc()
    print('\n>>> vLLM import FAILED — serving will stay analytic. This is common on T4 (sm 7.5).')

In [ ]:
# 2b · Smoke test: load TinyLlama and generate 8 tokens (surfaces load-time errors).
SMOKE_OK = False
if VLLM_OK:
    try:
        from vllm import LLM, SamplingParams
        _llm = LLM(model='TinyLlama/TinyLlama-1.1B-Chat-v1.0', enforce_eager=True,
                   dtype='half', gpu_memory_utilization=0.9, max_model_len=1024)
        _out = _llm.generate(['The capital of France is'], SamplingParams(max_tokens=8))
        print('SMOKE OK ->', repr(_out[0].outputs[0].text))
        SMOKE_OK = True
        del _llm
        import gc, torch; gc.collect(); torch.cuda.empty_cache()
    except Exception:
        import traceback; traceback.print_exc()
        print('\n>>> vLLM loaded but could not serve TinyLlama on this GPU.')
else:
    print('skipped — vLLM not importable')

In [ ]:
# 2c · Real serving benchmark (measures if the smoke test passed, else analytic).
from helix.inference import serving_benchmark, print_serving_result

# ModelConfig matching TinyLlama-1.1B so analytic vs measured is apples-to-apples.
tinyllama = ModelConfig(name='tinyllama-1.1b', num_layers=22, num_heads=32,
                        num_kv_heads=4, head_dim=64, ffn_dim=5632, vocab_size=32000)
model_arg = 'TinyLlama/TinyLlama-1.1B-Chat-v1.0' if SMOKE_OK else None
res = serving_benchmark(tinyllama, batch=8, prompt_len=512, gen_len=64,
                        device=DEVICE, model=model_arg, backend='auto')
print_serving_result(res)
print('\nRAW:', res.to_dict())
print('\n>>> backend =', res.backend, '("vllm" = real measurement, "analytic" = prediction)')

## 3 · Triton flash-attention — correctness + fair speed

Triton runs on Turing (T4) too, so this is **not** gated on Ampere.

In [ ]:
# 3a · Numerical correctness vs a full-precision reference.
import math, torch
from helix.kernels.attention_triton import flash_attention_triton, triton_available

if triton_available():
    try:
        torch.manual_seed(0)
        B,S,H,D = 4,512,16,64
        q,k,v = (torch.randn(B,S,H,D,device='cuda',dtype=torch.float16) for _ in range(3))
        def ref(q,k,v,causal):
            s = torch.einsum('bqhd,bkhd->bhqk', q.float(),k.float())/math.sqrt(D)
            if causal:
                m=torch.tril(torch.ones(S,S,device='cuda',dtype=torch.bool)); s=s.masked_fill(~m,float('-inf'))
            return torch.einsum('bhqk,bkhd->bqhd', torch.softmax(s,-1), v.float())
        for c in (False,True):
            e=(flash_attention_triton(q,k,v,causal=c).float()-ref(q,k,v,c)).abs().max().item()
            print(f'triton causal={c}: max abs err {e:.2e} -> {"PASS" if e<2e-2 else "FAIL"}')
    except Exception:
        import traceback; traceback.print_exc()
else:
    print('Triton unavailable (needs torch+triton on CUDA).')

In [ ]:
# 3b · Fair speed: torch SDPA (fp16) vs Triton (fp16), identical non-constant inputs.
import time, torch.nn.functional as F
if triton_available():
    try:
        B,S,H,D = 8,512,16,64
        q,k,v = (torch.randn(B,S,H,D,device='cuda',dtype=torch.float16) for _ in range(3))
        def sdpa(q,k,v):
            qs,ks,vs=(t.transpose(1,2) for t in (q,k,v))
            return F.scaled_dot_product_attention(qs,ks,vs).transpose(1,2)
        def bench(fn):
            for _ in range(10): fn(q,k,v); torch.cuda.synchronize()
            t=[]
            for _ in range(50):
                s=time.perf_counter(); fn(q,k,v); torch.cuda.synchronize(); t.append((time.perf_counter()-s)*1e3)
            return sum(t)/len(t)
        st, tt = bench(sdpa), bench(flash_attention_triton)
        print(f'torch SDPA  fp16 : {st:.3f} ms')
        print(f'triton flash fp16: {tt:.3f} ms   ({st/tt:.2f}x vs SDPA)')
        print('(SDPA is a vendor-tuned kernel; on T4 Triton being slower is expected + honest.)')
    except Exception:
        import traceback; traceback.print_exc()
else:
    print('skipped — Triton unavailable')

## 4 · Inference sweep — does an 8B fit, and where does it spill?

In [ ]:
from helix.inference import sweep_inference, print_sweep

print('### fp16 weights (16 GB) — expect it NOT to fit a 16 GB T4:')
print_sweep(sweep_inference(ModelConfig.from_preset('llama3-8b', dtype_bytes=2),
            batches=[1,8,32,64,128], prompt_lens=[2048], gen_len=128, device=DEVICE),
            'llama3-8b fp16', DEVICE)

print('### fp8 weights (8 GB) — frees HBM; watch HBM flip yes->no as batch grows:')
print_sweep(sweep_inference(ModelConfig.from_preset('llama3-8b', dtype_bytes=1),
            batches=[1,8,32,64,128], prompt_lens=[2048], gen_len=128, device=DEVICE),
            'llama3-8b fp8', DEVICE)

## 5 · Copy back to Claude

Paste the **entire** output of sections 2, 3, and 4 — including any tracebacks.
The key lines: `backend =` (2c), the Triton `PASS/FAIL` + timing (3), and both
sweep tables (4).